# FLUX.2 Live Demo with Gallium

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarsZDF/gallium/blob/main/flux2_demo.ipynb)

**Live image generation with BFL's FLUX.2 API + experiment tracking with Gallium.**

This notebook demonstrates:
- Generating images with FLUX.2 via the BFL API
- Tracking experiments with gallium
- Parameter sweeps (seed, guidance, model)
- Creating comparison grids

**Requirements:**
- BFL API key (get one at https://api.bfl.ml/)

## Installation

In [ ]:
# Install dependencies
!pip install -q elemental-gallium[grid] requests

## Configuration

Enter your BFL API key below. You can get one at https://api.bfl.ml/

In [ ]:
import os
from getpass import getpass

# Configuration
BFL_API_KEY = os.environ.get("BFL_API_KEY") or getpass("Enter your BFL API key: ")

# API endpoint - change this to use different models
# Options:
#   https://api.bfl.ai/v1/flux-2-klein-9b  (klein 9B - fastest)
#   https://api.bfl.ai/v1/flux-pro-1.1     (pro 1.1)
#   https://api.bfl.ai/v1/flux-pro         (pro)
#   https://api.bfl.ai/v1/flux-dev         (dev)
BFL_API_ENDPOINT = "https://api.bfl.ai/v1/flux-2-klein-9b"

print(f"Using endpoint: {BFL_API_ENDPOINT}")
print(f"API key configured: {'Yes' if BFL_API_KEY else 'No'}")

## Setup

In [ ]:
import base64
import time
from io import BytesIO
from pathlib import Path

import requests
from PIL import Image

import gallium
import gallium.flux as gf

# Create output directory
Path("outputs").mkdir(exist_ok=True)

# Initialize gallium
gallium.init("flux2_experiments.db")

print(f"Gallium version: {gallium.__version__}")
print(f"FLUX.2 models available: {gf.MODELS}")

## BFL API Client

In [ ]:
def generate_image(
    prompt: str,
    seed: int = 42,
    width: int = 1024,
    height: int = 1024,
    guidance: float = 7.5,
    steps: int = 28,
    api_key: str = BFL_API_KEY,
    endpoint: str = BFL_API_ENDPOINT,
) -> tuple[Image.Image, int]:
    """Generate an image using BFL's FLUX.2 API.
    
    Args:
        prompt: Text prompt for generation
        seed: Random seed for reproducibility
        width: Image width
        height: Image height
        guidance: Guidance scale (1.0-20.0)
        steps: Number of inference steps
        api_key: BFL API key
        endpoint: BFL API endpoint URL
        
    Returns:
        Tuple of (PIL Image, generation time in ms)
    """
    # BFL API uses x-key header for authentication
    headers = {
        "accept": "application/json",
        "x-key": api_key,
        "Content-Type": "application/json",
    }
    
    # Map width/height to aspect ratio for BFL API
    if width == height:
        aspect_ratio = "1:1"
    elif width > height:
        aspect_ratio = "16:9" if width / height > 1.5 else "4:3"
    else:
        aspect_ratio = "9:16" if height / width > 1.5 else "3:4"
    
    payload = {
        "prompt": prompt,
        "seed": seed,
        "aspect_ratio": aspect_ratio,
        "guidance_scale": guidance,
        "num_inference_steps": steps,
    }
    
    start_time = time.time()
    
    # Submit generation request
    response = requests.post(endpoint, headers=headers, json=payload)
    response.raise_for_status()
    result = response.json()
    
    # Get the polling URL from the response (required for BFL API)
    polling_url = result.get("polling_url")
    if not polling_url:
        raise ValueError(f"No polling_url in response: {result}")
    
    # Poll for results using the returned polling_url
    while True:
        time.sleep(0.5)
        
        status_response = requests.get(polling_url, headers={"accept": "application/json", "x-key": api_key})
        status_response.raise_for_status()
        status = status_response.json()
        
        if status.get("status") == "Ready":
            result = status
            break
        elif status.get("status") in ("Error", "Failed"):
            raise RuntimeError(f"Generation failed: {status}")
        
        # Print progress if available
        progress = status.get("progress")
        if progress is not None:
            print(f"  Progress: {progress:.0%}", end="\r")
    
    duration_ms = int((time.time() - start_time) * 1000)
    
    # Download image from URL (URLs expire after 10 minutes!)
    if "result" in result and "sample" in result["result"]:
        image_url = result["result"]["sample"]
        image_response = requests.get(image_url)
        image_response.raise_for_status()
        image = Image.open(BytesIO(image_response.content))
    else:
        raise ValueError(f"Unexpected response format: {result.keys()}")
    
    return image, duration_ms


def generate_and_track(
    prompt: str,
    seed: int = 42,
    width: int = 1024,
    height: int = 1024,
    guidance: float = 7.5,
    steps: int = 28,
    model: str = "flux.2-klein",
    output_dir: str = "outputs",
    **kwargs,
) -> gallium.Experiment:
    """Generate an image and track it with gallium.
    
    Returns:
        The logged Experiment object
    """
    # Generate image
    image, duration_ms = generate_image(
        prompt=prompt,
        seed=seed,
        width=width,
        height=height,
        guidance=guidance,
        steps=steps,
        **kwargs,
    )
    
    # Save image
    path = f"{output_dir}/{model.replace('.', '_')}_{seed}_{int(time.time())}.png"
    image.save(path)
    
    # Log to gallium
    exp_id = gallium.log(
        prompt=prompt,
        seed=seed,
        path=path,
        model=model,
        width=image.width,
        height=image.height,
        duration_ms=duration_ms,
        params={"guidance": guidance, "steps": steps},
    )
    
    print(f"Generated: {path} ({duration_ms}ms)")
    
    # Return the most recent experiment (the one we just logged)
    recent = gallium.recent(1)
    return recent[0] if recent else None

print("API client ready!")

## 1. Single Generation

Generate a single image and track it.

In [ ]:
# Generate a single image
exp = generate_and_track(
    prompt="A cyberpunk city at night, neon lights, rain-slicked streets, highly detailed",
    seed=42,
    guidance=7.5,
)

# Display the result
if exp and exp.path:
    img = Image.open(exp.path)
    display(img.resize((512, 512)))

## 2. Seed Sweep

Generate multiple variations with different seeds to find good compositions.

In [ ]:
# Define the prompt
prompt = "A magical forest with glowing mushrooms, ethereal atmosphere, fantasy art"

# Generate seed variations using gallium.flux helper
seeds = [42, 123, 456, 789]
params_list = gf.seed_sweep(prompt, seeds=seeds)

print(f"Generating {len(seeds)} seed variations...")
for params in params_list:
    generate_and_track(
        prompt=params["prompt"],
        seed=params["seed"],
        guidance=params["params"]["guidance"],
        steps=params["params"]["steps"],
        model="flux.2-klein",
    )

In [ ]:
# Create a comparison grid
forest_exps = gallium.find(prompt__contains="magical forest")
if forest_exps:
    grid = gf.sweep_grid(forest_exps, "seed", cols=2, max_size=512)
    grid.save("outputs/seed_sweep.png")
    display(grid)

## 3. Guidance Scale Sweep

Find the optimal guidance scale for prompt adherence vs. creativity.

In [ ]:
# Guidance sweep with the same seed
prompt = "A serene Japanese garden with cherry blossoms, koi pond, traditional architecture"
guidance_values = [3.0, 7.5, 12.0, 18.0]

params_list = gf.guidance_sweep(prompt, guidance_values=guidance_values, seed=42)

print(f"Generating {len(guidance_values)} guidance variations...")
for params in params_list:
    generate_and_track(
        prompt=params["prompt"],
        seed=params["seed"],
        guidance=params["params"]["guidance"],
        steps=params["params"]["steps"],
        model="flux.2-klein",
    )

In [ ]:
# Create guidance comparison grid
garden_exps = gallium.find(prompt__contains="Japanese garden")
if garden_exps:
    grid = gf.sweep_grid(garden_exps, "guidance", cols=4, max_size=256)
    grid.save("outputs/guidance_sweep.png")
    display(grid)

## 4. Star Your Favorites

Mark the best experiments and add notes.

In [ ]:
# View all experiments
all_exps = gallium.find()
print(f"Total experiments: {len(all_exps)}")
print("\nRecent experiments:")
for exp in gallium.recent(5):
    print(f"  [{exp.id}] {exp.prompt[:40]}... (seed={exp.seed}, {exp.duration_ms}ms)")

In [ ]:
# Star the best one (change the ID based on your results)
# best_id = 1  # Uncomment and set to your favorite
# gallium.star(best_id)
# gallium.annotate(best_id, "Best composition from the seed sweep")

# Show starred experiments
starred = gallium.find(starred=True)
print(f"Starred experiments: {len(starred)}")
for exp in starred:
    print(f"  [{exp.id}] {exp.prompt[:40]}... - {exp.notes}")

## 5. Matrix Grid

Compare experiments across two dimensions.

In [ ]:
# Create matrix comparing prompts vs seeds
all_exps = gallium.find()
if len(all_exps) >= 4:
    matrix = gallium.matrix_grid(
        all_exps,
        rows="prompt",
        cols="seed",
        max_size=200,
        show_labels=True,
    )
    matrix.save("outputs/matrix_comparison.png")
    display(matrix)

## 6. Export Results

In [ ]:
# Export to different formats
gallium.export("csv", path="outputs/experiments.csv")
gallium.export("json", path="outputs/experiments.json")
gallium.export("html", path="outputs/gallery.html", title="FLUX.2 Experiments")

print("Exported to:")
print("  - outputs/experiments.csv")
print("  - outputs/experiments.json")
print("  - outputs/gallery.html")

## Summary

This notebook demonstrated:

1. **BFL API Integration** - Generate images with FLUX.2 models
2. **Experiment Tracking** - Log all generations with metadata
3. **Parameter Sweeps** - Systematically explore seed and guidance values
4. **Visual Comparison** - Create grids and matrices for side-by-side comparison
5. **Curation** - Star and annotate your best results
6. **Export** - Save results to CSV, JSON, or HTML gallery

### Next Steps

- Try different prompts and parameters
- Experiment with aspect ratio sweeps (`gf.aspect_ratio_sweep`)
- Compare different BFL API endpoints by changing `BFL_API_ENDPOINT`
- Check out the [examples/](https://github.com/MarsZDF/gallium/tree/main/examples) for more integration patterns